### **Convert raw NEMO files to NEMO T,S,melt on grid**

In [1]:
import glob
import os
import numpy as np
import xarray as xr
import time

In [2]:
import numpy as np
import xarray as xr    
#import cartopy.crs as ccrs
#import matplotlib.ticker as mticker
#import matplotlib.pyplot as plt
#from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
# To interpolate NEMO data from Nico's mask files to the NEMO simulation grid 
from scipy.interpolate import griddata
import itertools
# To convert coordinates from lat, lon to polar stereo x,y 
from pyproj import Transformer
# KD Tree for nearest neighbours
from scipy.spatial import cKDTree
# To play a sound when at the end of the code you want to run 
import os
##os.system("printf '\a'")
import time
import glob

In [3]:
# Set all the filepaths that will be required 
filepath_base = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/'
filepath_nn_input = filepath_base + "data/processing_ho/nn_input_"

In [4]:
nemo_runs_all = 'OPM016_OPM018_OPM021_ctrl94_isf94_isfru94'
# Options are 'OPM016', 'OPM018', 'OPM021', 'ctrl94', 'isf94', 'isfru94'

In [5]:
nemo_runs = nemo_runs_all.split('_')
nemo_runs

['OPM016', 'OPM018', 'OPM021', 'ctrl94', 'isf94', 'isfru94']

In [15]:
for jj in nemo_runs:
    nemo_run = jj
    
    # Set all the filepaths that will be required 
    if nemo_run == 'OPM016':
        filepath_nemo = '/bettik/burgardc/DATA/BASAL_MELT_PARAM/raw/NEMO_eORCA025.L121-OPM016/'
        file_extension = '*gridT*.nc'
        filepath_mesh_mask = filepath_nemo + 'eORCA025.L121-' + nemo_run + '_mesh_mask.nc'
        monthly = False
    elif nemo_run == 'OPM018':
        filepath_nemo = '/bettik/burgardc/DATA/BASAL_MELT_PARAM/raw/NEMO_eORCA025.L121-OPM018/'
        file_extension = '*gridT*.nc'
        filepath_mesh_mask = filepath_nemo + 'eORCA025.L121-' + nemo_run + '_mesh_mask.nc'
        monthly = False
    elif nemo_run == 'OPM021':
        filepath_nemo = '/bettik/burgardc/DATA/BASAL_MELT_PARAM/raw/NEMO_eORCA025.L121-OPM021/'
        file_extension = '*gridT*.nc'
        filepath_mesh_mask = filepath_nemo + 'eORCA025.L121-' + nemo_run + '_mesh_mask.nc'
        monthly = False
    elif nemo_run == 'ctrl94':
        filepath_nemo = '/bettik/burgardc/DATA/SUMMER_PAPER/raw/CHRISTOPH_DATA/'
        file_extension = 'fwfisf*' + nemo_run + '*.nc'
        filepath_mesh_mask = filepath_nemo + 'mesh_mask.nc'
        filepath_cfg = filepath_nemo + 'domain_cfg_eANT025.L121.nc'
        monthly = False
    elif nemo_run == 'isf94':
        filepath_nemo = '/bettik/burgardc/DATA/SUMMER_PAPER/raw/CHRISTOPH_DATA/'
        file_extension = 'fwfisf*' + nemo_run + '*.nc'
        filepath_mesh_mask = filepath_nemo + 'mesh_mask.nc'
        filepath_cfg = filepath_nemo + 'domain_cfg_eANT025.L121.nc'
        monthly = False
    elif nemo_run == 'isfru94':
        filepath_nemo = '/bettik/burgardc/DATA/SUMMER_PAPER/raw/CHRISTOPH_DATA/'
        file_extension = 'fwfisf*' + nemo_run + '*.nc'
        filepath_mesh_mask = filepath_nemo + 'mesh_mask.nc'
        filepath_cfg = filepath_nemo + 'domain_cfg_eANT025.L121.nc'
        monthly = False
    else:
        filepath_pierre = []
        file_extension = []
        print('Help, I do not know this nemo_run')
    #print(nemo_run, filepath_nemo)
    
    filepaths = glob.glob(filepath_nemo + file_extension)
    years_to_run = []
    if monthly == True:
        months = []
    if monthly == False:
        if nemo_run in ('OPM016', 'OPM018', 'OPM021'):
            for i in range(len(filepaths)):
                year = filepaths[i].split(os.path.sep)[-1].split('_')[1].split('.')[0].split('y')[1]
                years_to_run.append(year)
        if nemo_run in ('ctrl94', 'isf94', 'isfru94'):
            yr0, yrfin = filepaths[0].split(os.path.sep)[-1].split('_')[1].split('.')[0].split('-')
            yr0, yrfin
            yrlist = np.arange(int(yr0), int(yrfin)+1, 1)
            for i in range(len(yrlist)):
                years_to_run.append(str(yrlist[i]))
    years_to_run.sort()
    #print(len(np.unique(years_to_run)), 'files to be processed')
    
    filepath_run = glob.glob(filepath_nn_input + nemo_run +'_y' +  '*.nc')
    years_run = []
    for i in range(len(filepath_run)):
        year = (filepath_run[i].split(os.path.sep)[-1].split('_')[-1].split('.')[0].split('y')[-1])
        years_run.append(year)
    years_run.sort()
    #print(years_run)
    print(len(years_run), 'files processed')
    
    still_to_run = []
    for i in years_to_run:
        if i in years_run:
            []
        else:
            still_to_run.append(i)
    print(nemo_run, 'These years still to be processed:', still_to_run)

29 files processed
OPM016 These years still to be processed: []
28 files processed
OPM018 These years still to be processed: ['1999']
30 files processed
OPM021 These years still to be processed: []
32 files processed
ctrl94 These years still to be processed: []
87 files processed
isf94 These years still to be processed: []
87 files processed
isfru94 These years still to be processed: []
